In [158]:
"""
ai_picks_pipeline.py
 
Ensemble AI horse racing picks (Claude / Gemini) with morning line odds,
convert to win probabilities, and prepare for backtesting.
 
Data structure:
    data/ai_picks/{date}/claude.csv     - AI estimated Beyers
    data/ai_picks/{date}/gemini.csv     - AI estimated Beyers
    data/upc_entries/{track}.csv        - Entries with ML odds
    data/entries/*.csv                  - Historical results
 
AI pick CSV format:
    race, horse_number, expected_beyer, track
 
Results CSV format (data/entries/*.csv):
    running, horse, horse_name, sire, trainer, jockey, ml_odds,
    finish, win_paid, place_paid, show_paid, track, date, race
"""
 
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import log_loss
 
 

In [159]:
# =============================================================================
# 1. LOADING
# =============================================================================
 
def load_ai_picks(date_str, base_dir='../data/ai_picks'):
    """
    Load Claude and Gemini pick CSVs for a given date.
    Each CSV has: race, horse_number, expected_beyer, track
    
    Returns merged DataFrame with claude_beyer, gemini_beyer columns.
    """
    date_dir = os.path.join(base_dir, date_str)
    
    picks = {}
    for source in ['claude', 'gemini']:
        path = os.path.join(date_dir, f'{source}.csv')
        if os.path.exists(path):
            df = pd.read_csv(path)
            
            # Normalize column names - find the beyer column
            cols = df.columns.tolist()
            beyer_col = [c for c in cols if 'beyer' in c.lower() or 'expected' in c.lower()]
            if beyer_col:
                df = df.rename(columns={beyer_col[0]: f'{source}_beyer'})
            else:
                # If no track column, assume 3 cols: race, horse_number, beyer
                if len(cols) == 3:
                    df.columns = ['race', 'horse_number', f'{source}_beyer']
                elif len(cols) == 4:
                    df.columns = ['race', 'horse_number', f'{source}_beyer', 'track']
            
            df['race'] = df['race'].astype(int)
            df['horse_number'] = df['horse_number'].astype(int)
            
            keep_cols = ['race', 'horse_number', f'{source}_beyer']
            if 'track' in df.columns:
                keep_cols.append('track')
            
            picks[source] = df[keep_cols]
    
    if not picks:
        raise FileNotFoundError(f"No AI pick files found in {date_dir}")
    
    # Merge on race + horse_number (and track if present)
    sources = list(picks.keys())
    merged = picks[sources[0]]
    for source in sources[1:]:
        merge_keys = ['race', 'horse_number']
        if 'track' in merged.columns and 'track' in picks[source].columns:
            merge_keys.append('track')
        merged = merged.merge(picks[source], on=merge_keys, how='outer')
    
    merged['date'] = date_str
    
    # Default track if not present
    if 'track' not in merged.columns:
        merged['track'] = 'keeneland'  # sensible default for Kee card
    
    return merged
 

In [160]:
def load_entries(base_dir='../data/entries', track=None, date=None):
    """
    Load historical results from data/entries/*.csv
    
    Format: running, horse, horse_name, sire, trainer, jockey, ml_odds,
            finish, win_paid, place_paid, show_paid, track, date, race
    """
    results_files = [f for f in os.listdir(base_dir) if f.endswith('.csv')]
    
    if not results_files:
        return pd.DataFrame()
    
    dfs = []
    for f in results_files:
        try:
            df = pd.read_csv(os.path.join(base_dir, f))
            dfs.append(df)
        except Exception as e:
            print(f"Warning: Could not load {f}: {e}")
    
    if not dfs:
        return pd.DataFrame()
    
    results = pd.concat(dfs, ignore_index=True)
    
    # Only running horses
    if 'running' in results.columns:
        results = results[results['running'] == 1].copy()
    
    # Parse date
    if 'date' in results.columns:
        results['date'] = pd.to_datetime(results['date']).dt.strftime('%Y-%m-%d')
    
    # Filter
    if track is not None:
        results = results[results['track'] == track]
    if date is not None:
        results = results[results['date'] == date]
    
    # Derived columns
    if 'finish' in results.columns:
        results['won'] = (results['finish'] == 1).astype(int)
        results['place'] = (results['finish'] <= 2).astype(int)
        results['show'] = (results['finish'] <= 3).astype(int)
    
    # Payout per dollar
    if 'win_paid' in results.columns:
        results['payout_per_dollar'] = pd.to_numeric(
            results['win_paid'].astype(str).str.replace('$', '').str.replace(',', ''),
            errors='coerce'
        ).fillna(0) / 2  # $2 base
    
    # Create horse_number from 'horse' column if needed
    if 'horse' in results.columns and 'horse_number' not in results.columns:
        results['horse_number'] = results['horse'].astype(int)
    
    return results.reset_index(drop=True)
 
 

In [161]:
def load_upc_entries(track, base_dir='../data/upc_entries'):
    """
    Load upcoming/current entries with ML odds from data/upc_entries/{track}.csv
    """
    path = os.path.join(base_dir, f'{track}.csv')
    if not os.path.exists(path):
        return None
    
    df = pd.read_csv(path)
    return df
 

In [162]:
def load_all_ai_dates(base_dir='data/ai_picks'):
    """Load and concatenate all available AI pick dates."""
    if not os.path.exists(base_dir):
        return pd.DataFrame()
    
    dates = sorted([d for d in os.listdir(base_dir) 
                    if os.path.isdir(os.path.join(base_dir, d))])
    
    all_picks = []
    for date_str in dates:
        try:
            picks = load_ai_picks(date_str, base_dir)
            all_picks.append(picks)
        except FileNotFoundError:
            continue
    
    return pd.concat(all_picks, ignore_index=True) if all_picks else pd.DataFrame()
 
 

In [163]:
# =============================================================================
# 2. BEYER COMPOSITE & BLENDING
# =============================================================================
 
def compute_composite_beyer(df, weights=None):
    """
    Blend multiple Beyer estimates into a single composite.
    Uses confidence-adaptive weighting based on source disagreement.
    
    Default weights (no ML):  claude: 0.55, gemini: 0.45
    With ML:                  ml: 0.45, claude: 0.30, gemini: 0.25
    """
    beyer_cols = [c for c in df.columns if c.endswith('_beyer') and 
                  c not in ('composite_beyer', 'beyer_disagreement')]
    
    if not beyer_cols:
        raise ValueError("No Beyer columns found in DataFrame")
    
    if weights is None:
        if 'ml_beyer' in beyer_cols:
            weights = {'ml_beyer': 0.45, 'claude_beyer': 0.30, 'gemini_beyer': 0.25}
        elif len(beyer_cols) == 2:
            weights = {'claude_beyer': 0.55, 'gemini_beyer': 0.45}
        else:
            weights = {beyer_cols[0]: 1.0}
    
    # Only use available columns
    available = {k: v for k, v in weights.items() if k in df.columns}
    total_w = sum(available.values())
    available = {k: v / total_w for k, v in available.items()}
    
    result_df = df.copy()
    
    if len(available) >= 2:
        beyer_matrix = result_df[list(available.keys())].values
        row_std = np.nanstd(beyer_matrix, axis=1)
        result_df['beyer_disagreement'] = row_std
        
        LOW_THRESH = 5.0
        HIGH_THRESH = 15.0
        
        composites = np.zeros(len(result_df))
        for i in range(len(result_df)):
            if row_std[i] < LOW_THRESH:
                w = {k: 1.0 / len(available) for k in available}
            elif row_std[i] > HIGH_THRESH:
                if 'ml_beyer' in available:
                    w = {'ml_beyer': 0.55, 'claude_beyer': 0.25, 'gemini_beyer': 0.20}
                else:
                    w = {'claude_beyer': 0.60, 'gemini_beyer': 0.40}
                w = {k: v for k, v in w.items() if k in available}
            else:
                w = available.copy()
            
            tw = sum(w.values())
            vals = sum(result_df.iloc[i][col] * (w[col] / tw)
                       for col in w 
                       if col in result_df.columns and not pd.isna(result_df.iloc[i][col]))
            composites[i] = vals
        
        result_df['composite_beyer'] = composites
    else:
        col = list(available.keys())[0]
        result_df['composite_beyer'] = result_df[col]
        result_df['beyer_disagreement'] = 0.0
    
    return result_df

In [164]:
# =============================================================================
# 3. BEYER -> WIN PROBABILITY (t-distribution Monte Carlo)
# =============================================================================
 
def beyer_to_win_probabilities(df, composite_col='composite_beyer',
                                std_dev=4.8354, degrees_of_freedom=0.8108,
                                num_simulations=100000,
                                group_cols=None, seed=42):
    """
    Convert composite Beyer ratings to win probabilities using
    t-distribution Monte Carlo simulation.
    
    From backtest notebook Cell 7, optimized params from Cell 9:
        std_dev=4.8354, degrees_of_freedom=0.8108
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    # Use only columns that exist
    group_cols = [c for c in group_cols if c in df.columns]
    
    result_df = df.copy()
    result_df['projected_win_probability'] = 0.0
    
    rng = np.random.RandomState(seed)
    t_dist = stats.t(df=degrees_of_freedom)
    
    for group_vals, race_df in result_df.groupby(group_cols):
        race_indices = race_df.index
        num_horses = len(race_indices)
        
        if num_horses <= 1:
            result_df.loc[race_indices, 'projected_win_probability'] = 1.0
            continue
        
        composite_values = race_df[composite_col].values.reshape(num_horses, 1)
        random_effects = t_dist.rvs(size=(num_horses, num_simulations),
                                     random_state=rng) * std_dev
        finishes = composite_values + random_effects
        winners = np.argmax(finishes, axis=0)
        win_counts = np.bincount(winners, minlength=num_horses)
        win_probs = np.maximum(win_counts / num_simulations, 1e-6)
        win_probs = win_probs / win_probs.sum()
        
        result_df.loc[race_indices, 'projected_win_probability'] = win_probs
    
    return result_df

In [165]:
# =============================================================================
# 4. MORNING LINE PROCESSING
# =============================================================================
 
def parse_ml_odds(odds_str):
    """Parse fractional odds string to implied probability."""
    try:
        if pd.isna(odds_str) or str(odds_str).strip() == '':
            return np.nan
        odds_str = str(odds_str).strip()
        if '/' in odds_str:
            num, den = map(float, odds_str.split('/'))
            return den / (num + den)
        elif odds_str.upper() in ('EVN', 'EVEN', 'EV'):
            return 0.5
        else:
            decimal_odds = float(odds_str)
            if decimal_odds > 0:
                return 1 / (decimal_odds + 1)  # American-style: 5/1 stored as 5
            return np.nan
    except (ValueError, ZeroDivisionError):
        return np.nan
 
 
def process_morning_lines(df, ml_odds_col='ml_odds',
                          group_cols=None, temperature=1.0):
    """
    Convert morning line odds to normalized probabilities using
    softmax with temperature scaling.
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    group_cols = [c for c in group_cols if c in df.columns]
    
    result_df = df.copy()
    result_df['ml_probability_raw'] = result_df[ml_odds_col].apply(parse_ml_odds)
    result_df['ml_probability_normalized'] = np.nan
    
    for group_vals, group_df in result_df.groupby(group_cols):
        group_indices = group_df.index
        valid_mask = ~group_df['ml_probability_raw'].isna()
        valid_indices = group_indices[valid_mask]
        
        if len(valid_indices) > 0:
            valid_probs = group_df.loc[valid_indices, 'ml_probability_raw'].values
            logits = np.log(np.maximum(valid_probs, 1e-10)) / temperature
            exp_logits = np.exp(logits - np.max(logits))
            normalized = exp_logits / exp_logits.sum()
            result_df.loc[valid_indices, 'ml_probability_normalized'] = normalized
    
    return result_df

In [166]:
# =============================================================================
# 5. PARAMETER OPTIMIZATION (requires historical results)
# =============================================================================
 
def optimize_simulation_params(df, composite_col='composite_beyer',
                                group_cols=None,
                                initial_std=4.8, initial_df=0.8,
                                num_simulations=10000):
    """
    Optimize std_dev and degrees_of_freedom to minimize log loss.
    Requires 'won' column (1 = winner, 0 = loser).
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    
    if 'won' not in df.columns:
        raise ValueError("Need 'won' column for optimization")
    
    def objective(params):
        std_dev, deg_free = params
        result = beyer_to_win_probabilities(
            df, composite_col=composite_col,
            std_dev=std_dev, degrees_of_freedom=deg_free,
            num_simulations=num_simulations,
            group_cols=group_cols
        )
        probs = np.clip(result['projected_win_probability'].values, 1e-6, 1 - 1e-6)
        return log_loss(result['won'].values, probs)
    
    result = minimize(
        objective,
        x0=np.array([initial_std, initial_df]),
        method='Nelder-Mead',
        options={'maxiter': 250, 'disp': True}
    )
    
    optimal_std, optimal_df = result.x
    print(f"Optimal std_dev: {optimal_std:.4f}")
    print(f"Optimal degrees_of_freedom: {optimal_df:.4f}")
    print(f"Final log loss: {result.fun:.6f}")
    
    return optimal_std, optimal_df, result.fun
 
 
def optimize_blend_weights(df, group_cols=None,
                           std_dev=4.8354, degrees_of_freedom=0.8108,
                           num_simulations=10000):
    """
    Optimize Claude / Gemini / ML blend weights via log loss.
    Uses softmax parameterization to keep weights positive and sum to 1.
    """
    if group_cols is None:
        group_cols = ['date', 'track', 'race']
    
    beyer_cols = [c for c in ['claude_beyer', 'gemini_beyer', 'ml_beyer']
                  if c in df.columns]
    
    if len(beyer_cols) < 2:
        raise ValueError(f"Need ≥2 Beyer sources, found: {beyer_cols}")
    
    def objective(raw_weights):
        weights = np.exp(raw_weights) / np.sum(np.exp(raw_weights))
        weight_dict = dict(zip(beyer_cols, weights))
        
        test_df = df.copy()
        test_df['composite_beyer'] = sum(
            test_df[col] * w for col, w in weight_dict.items()
        )
        
        result = beyer_to_win_probabilities(
            test_df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
            num_simulations=num_simulations, group_cols=group_cols
        )
        probs = np.clip(result['projected_win_probability'].values, 1e-6, 1 - 1e-6)
        return log_loss(result['won'].values, probs)
    
    result = minimize(objective, np.zeros(len(beyer_cols)),
                      method='Nelder-Mead', options={'maxiter': 200, 'disp': True})
    
    optimal_weights = np.exp(result.x) / np.sum(np.exp(result.x))
    weight_dict = dict(zip(beyer_cols, optimal_weights))
    
    print("\nOptimal blend weights:")
    for col, w in weight_dict.items():
        print(f"  {col}: {w:.4f}")
    print(f"Final log loss: {result.fun:.6f}")
    
    return weight_dict, result.fun
 
 

In [167]:
# =============================================================================
# 6. EDGE CALCULATION
# =============================================================================
 
def calculate_edge(df, projected_prob_col='projected_win_probability',
                   ml_prob_col='ml_probability_normalized'):
    """Calculate edge, Kelly, expected ROI between model and ML."""
    result_df = df.copy()
    
    ml = result_df[ml_prob_col]
    proj = result_df[projected_prob_col]
    
    result_df['edge'] = proj - ml
    result_df['percentage_edge'] = np.where(ml > 0, ((proj / ml) - 1) * 100, np.nan)
    
    result_df['kelly_stake'] = np.where(
        ml > 0,
        np.clip((proj * (1 / ml) - 1) / (1 / ml - 1), 0, 1),
        np.nan
    )
    result_df['half_kelly'] = result_df['kelly_stake'] / 2
    
    result_df['expected_roi'] = np.where(ml > 0, (proj / ml) - 1, np.nan)
    
    return result_df
 

In [168]:
# =============================================================================
# 7. FULL PIPELINE
# =============================================================================
 
def run_pipeline(date_str, track='keeneland',
                 base_dir='../data/ai_picks',
                 entries_dir='../data/entries',
                 upc_dir='../data/upc_entries',
                 std_dev=4.8354, degrees_of_freedom=0.8108,
                 blend_weights=None, ml_temperature=1.0,
                 num_simulations=100000):
    """
    Full pipeline for a single race date:
    1. Load AI picks (Claude, Gemini)
    2. Compute composite Beyer
    3. Convert to win probabilities via MC simulation
    4. Load ML odds from upc_entries or entries
    5. Calculate edges
    6. Merge results if available
    """
    group_cols = ['date', 'track', 'race']
    
    # --- Step 1: Load AI picks ---
    print(f"Loading AI picks for {date_str}...")
    df = load_ai_picks(date_str, base_dir)
    
    # Ensure track column
    if 'track' not in df.columns:
        df['track'] = track
    
    print(f"  {len(df)} entries across {df['race'].nunique()} races")
    
    # --- Step 2: Composite Beyer ---
    print("Computing composite Beyer estimates...")
    df = compute_composite_beyer(df, weights=blend_weights)
    
    # --- Step 3: Win probabilities ---
    print(f"Running MC simulation (n={num_simulations:,}, "
          f"std={std_dev:.4f}, df={degrees_of_freedom:.4f})...")
    df = beyer_to_win_probabilities(
        df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
        num_simulations=num_simulations, group_cols=group_cols
    )
    
    # --- Step 4: Morning line ---
    # Try UPC entries first, then historical entries
    ml_merged = False
    
    upc = load_upc_entries(track, upc_dir)
    if upc is not None and 'ml_odds' in upc.columns:
        print(f"Loading ML odds from upc_entries/{track}.csv...")
        merge_cols = ['race', 'horse_number']
        if 'horse' in upc.columns and 'horse_number' not in upc.columns:
            upc['horse_number'] = upc['horse'].astype(int)
        
        upc_subset = upc[['race', 'horse_number', 'ml_odds']].drop_duplicates()
        df = df.merge(upc_subset, on=merge_cols, how='left')
        ml_merged = True
    
    if not ml_merged:
        # Try entries dir
        date_fmt = pd.to_datetime(date_str.replace('_', '-')).strftime('%Y-%m-%d') \
            if '_' in date_str else date_str
        entries = load_entries(entries_dir, track=track, date=date_fmt)
        if not entries.empty and 'ml_odds' in entries.columns:
            print(f"Loading ML odds from entries...")
            entries_subset = entries[['race', 'horse_number', 'ml_odds']].drop_duplicates()
            df = df.merge(entries_subset, on=['race', 'horse_number'], how='left')
            ml_merged = True
    
    if ml_merged and 'ml_odds' in df.columns:
        df = process_morning_lines(df, ml_odds_col='ml_odds',
                                    group_cols=group_cols,
                                    temperature=ml_temperature)
        print("Calculating edges...")
        df = calculate_edge(df)
    
    # --- Step 5: Results ---
    date_fmt = date_str
    if '_' in date_str:
        # Convert 4_3_26 -> 2026-04-03
        parts = date_str.split('_')
        if len(parts) == 3:
            m, d, y = parts
            y = f"20{y}" if len(y) == 2 else y
            date_fmt = f"{y}-{int(m):02d}-{int(d):02d}"
    
    entries = load_entries(entries_dir, track=track, date=date_fmt)
    if not entries.empty:
        print("Merging results for backtesting...")
        result_cols = ['race', 'horse_number', 'finish', 'won',
                       'win_paid', 'place_paid', 'show_paid', 'payout_per_dollar']
        result_cols = [c for c in result_cols if c in entries.columns]
        
        # Avoid duplicate ml_odds
        if 'ml_odds' in entries.columns and 'ml_odds' in df.columns:
            result_cols = [c for c in result_cols if c != 'ml_odds']
        
        df = df.merge(entries[result_cols].drop_duplicates(),
                      on=['race', 'horse_number'], how='left')
    
    # --- Summary ---
    beyer_cols = [c for c in df.columns if c.endswith('_beyer')
                  and c not in ('composite_beyer', 'beyer_disagreement')]
    print(f"\nPipeline complete: {len(df)} horses, {df['race'].nunique()} races")
    print(f"  Sources: {beyer_cols}")
    print(f"  ML: {'ml_probability_normalized' in df.columns}")
    print(f"  Results: {'won' in df.columns}")
    
    return df
 

In [169]:
def run_backtest(base_dir='../data/ai_picks', entries_dir='../data/entries',
                 std_dev=4.8354, degrees_of_freedom=0.8108,
                 optimize=False, num_simulations=50000):
    """
    Backtest across all available dates with results.
    Optionally re-optimize simulation params and blend weights.
    """
    group_cols = ['date', 'track', 'race']
    
    picks_df = load_all_ai_dates(base_dir)
    if picks_df.empty:
        print("No AI picks found.")
        return None
    
    # Load all entries/results
    entries = load_entries(entries_dir)
    if entries.empty:
        print("No results data. Cannot backtest yet.")
        return None
    
    # Merge
    df = picks_df.merge(entries, on=['date', 'track', 'race', 'horse_number'], how='inner')
    
    if df.empty or 'won' not in df.columns:
        print("No matching picks+results found.")
        return None
    
    print(f"Backtest: {len(df)} entries, {df['date'].nunique()} dates")
    
    # Composite
    df = compute_composite_beyer(df)
    
    if optimize:
        print("\nOptimizing simulation parameters...")
        std_dev, degrees_of_freedom, _ = optimize_simulation_params(
            df, num_simulations=num_simulations // 5, group_cols=group_cols
        )
        
        beyer_cols = [c for c in df.columns if c.endswith('_beyer')
                      and c not in ('composite_beyer', 'beyer_disagreement')]
        if len(beyer_cols) >= 2:
            print("\nOptimizing blend weights...")
            weights, _ = optimize_blend_weights(
                df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
                num_simulations=num_simulations // 5, group_cols=group_cols
            )
    
    # Final run
    df = beyer_to_win_probabilities(
        df, std_dev=std_dev, degrees_of_freedom=degrees_of_freedom,
        num_simulations=num_simulations, group_cols=group_cols
    )
    
    # Evaluate
    probs = np.clip(df['projected_win_probability'].values, 1e-6, 1 - 1e-6)
    loss = log_loss(df['won'].values, probs)
    print(f"\nLog loss: {loss:.6f}")
    
    # Calibration
    df['prob_bucket'] = pd.cut(
        df['projected_win_probability'],
        bins=[0, 0.05, 0.10, 0.20, 0.35, 1.0],
        labels=['<5%', '5-10%', '10-20%', '20-35%', '>35%']
    )
    cal = df.groupby('prob_bucket').agg(
        n=('won', 'count'),
        actual=('won', 'mean'),
        predicted=('projected_win_probability', 'mean')
    )
    print("\nCalibration:")
    print(cal)
    
    return df

In [170]:
# =============================================================================
# 8. REPORTING
# =============================================================================
 
def print_race_report(df, race_num):
    """Formatted report for a single race."""
    race = df[df['race'] == race_num].sort_values('projected_win_probability', ascending=False)
    
    print(f"\n{'='*72}")
    print(f"RACE {race_num}")
    print(f"{'='*72}")
    
    beyer_cols = [c for c in race.columns if c.endswith('_beyer')
                  and c not in ('composite_beyer', 'beyer_disagreement')]
    
    header = f"{'#':>3} {'Comp':>5} "
    for col in beyer_cols:
        header += f"{col.replace('_beyer','')[:6]:>6} "
    header += f"{'WinP':>7} "
    
    if 'ml_probability_normalized' in race.columns:
        header += f"{'ML%':>6} {'Edge%':>7} {'Kelly':>6}"
    if 'beyer_disagreement' in race.columns:
        header += f" {'Disg':>5}"
    
    print(header)
    print("-" * len(header))
    
    for _, row in race.iterrows():
        line = f"{int(row['horse_number']):>3} {row['composite_beyer']:>5.1f} "
        for col in beyer_cols:
            v = row.get(col, np.nan)
            line += f"{v:>6.1f} " if not pd.isna(v) else f"{'—':>6} "
        line += f"{row['projected_win_probability']:>6.1%} "
        
        if 'ml_probability_normalized' in race.columns:
            ml = row.get('ml_probability_normalized', np.nan)
            edge = row.get('percentage_edge', np.nan)
            kelly = row.get('kelly_stake', np.nan)
            line += f"{ml:>5.1%} " if not pd.isna(ml) else f"{'—':>5} "
            line += f"{edge:>+6.1f}% " if not pd.isna(edge) else f"{'—':>7} "
            line += f"{kelly:>5.3f}" if not pd.isna(kelly) else f"{'—':>5}"
        if 'beyer_disagreement' in race.columns:
            line += f" {row['beyer_disagreement']:>5.1f}"
        
        print(line)
 
 
def percentage_to_american_odds(probability):
    """Convert win probability to American odds."""
    if probability <= 0 or probability >= 1:
        return None
    if probability >= 0.5:
        return int(-100 * probability / (1 - probability))
    else:
        return int(100 * (1 - probability) / probability)
 

In [171]:
 
# =============================================================================
# MAIN
# =============================================================================
 
if __name__ == '__main__':
    
    DATE = '4_3_26'
    TRACK = 'keeneland'
    
    # Defaults from historical optimization (backtest notebook Cell 9)
    # Will be re-optimized once AI pick results accumulate
    STD_DEV = 4
    DEGREES_OF_FREEDOM = 1.5
    
    try:
        df = run_pipeline(
            DATE, track=TRACK,
            std_dev=STD_DEV,
            degrees_of_freedom=DEGREES_OF_FREEDOM,
            num_simulations=100000
        )
        
        for race_num in sorted(df['race'].unique()):
            print_race_report(df, race_num)
        
        # Save
        out_dir = os.path.join('../data/ai_picks', DATE)
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(os.path.join(out_dir, 'ensemble_output.csv'), index=False)
        print(f"\nSaved to {out_dir}/ensemble_output.csv")
        
    except FileNotFoundError as e:
        print(f"Setup needed: {e}")
        print(f"\n1. mkdir -p data/ai_picks/{DATE}")
        print(f"2. Place claude.csv (and gemini.csv) there")
        print(f"3. Re-run")
    
    # Uncomment when results exist:
    # run_backtest(optimize=True)

Loading AI picks for 4_3_26...
  110 entries across 10 races
Computing composite Beyer estimates...
Running MC simulation (n=100,000, std=4.0000, df=1.5000)...
Loading ML odds from upc_entries/keeneland.csv...
Calculating edges...
Merging results for backtesting...

Pipeline complete: 110 horses, 10 races
  Sources: ['claude_beyer', 'gemini_beyer']
  ML: True
  Results: True

RACE 1
  #  Comp claude gemini    WinP    ML%   Edge%  Kelly  Disg
-----------------------------------------------------------
  6  71.0   68.0   74.0  52.1% 19.7% +165.0% 0.404   3.0
  8  67.5   65.0   70.0  26.3%     —       —     —   2.5
  2  60.0   58.0   62.0   6.9% 26.2%  -73.6% 0.000   2.0
  5  57.5   55.0   60.0   5.0% 14.3%  -65.1% 0.000   2.5
  1  50.0   45.0   56.0   2.6% 11.2%  -77.2% 0.000   5.5
  3  46.5   42.0   52.0   2.0%  6.1%  -66.4% 0.000   5.0
  7  46.0   48.0   44.0   2.0%     —       —     —   2.0
  4  42.5   38.0   48.0   1.6% 22.5%  -92.8% 0.000   5.0
  9  39.0   30.0   50.0   1.4%     —  

In [150]:
import numpy as np
import pandas as pd
from operator import itemgetter
# from ai_picks_pipeline import (
#     load_ai_picks, compute_composite_beyer, beyer_to_win_probabilities,
#     process_morning_lines, print_race_report
# )
 
 

In [151]:
# =============================================================================
# PICK TICKET OPTIMIZER
# =============================================================================
 
def optimize_pick_tickets(df, race_numbers, bet_type=None, blend_factor=0.5,
                          pool_size=100000, takeout_rate=0.22,
                          max_cost=None, min_prob=None, min_total_prob=None,
                          budget_steps=20,
                          projected_prob_col='projected_win_probability',
                          ml_prob_col=None,
                          race_col='race', horse_col='horse_number',
                          debug=False):
    """
    Optimize Pick N tickets using model probabilities (optionally blended with ML).
    
    Parameters
    ----------
    df : DataFrame with race data
    race_numbers : list of race numbers to include
    bet_type : int, auto-detected from len(race_numbers) if None
    blend_factor : float, weight on model (1.0 = all model, 0.0 = all ML)
    pool_size : estimated pool size in dollars
    takeout_rate : track takeout (0.22 = 22%)
    max_cost : max ticket cost in combinations
    min_prob : min coverage per race (e.g. 0.7 = 70%)
    min_total_prob : min total hit probability
    budget_steps : number of frontier points to return
    
    Returns
    -------
    list of (cost, ev, probability, selections_dict)
    """
    if bet_type is None:
        bet_type = len(race_numbers)
    
    # Filter to requested races
    df_pick = df[df[race_col].isin(race_numbers)].copy()
    
    # Determine if we have ML probabilities
    has_ml = (ml_prob_col is not None and 
              ml_prob_col in df_pick.columns and 
              df_pick[ml_prob_col].notna().any())
    
    if not has_ml:
        # No ML: use projected prob as both model and "market"
        # This means EV calculation is neutral — we're just optimizing coverage
        blend_factor = 1.0
        ml_prob_col_used = projected_prob_col
        print("No ML odds available. Using model probabilities for ticket construction.")
        print("EV calculations will assume fair market (no overlay detection).")
    else:
        ml_prob_col_used = ml_prob_col
    
    # Calculate blended probabilities
    df_pick['blended_prob'] = (blend_factor * df_pick[projected_prob_col] + 
                                (1 - blend_factor) * df_pick[ml_prob_col_used])
    
    # Normalize within each race
    for race_num in race_numbers:
        mask = df_pick[race_col] == race_num
        total = df_pick.loc[mask, 'blended_prob'].sum()
        if total > 0:
            df_pick.loc[mask, 'blended_prob'] /= total
    
    # Build probability dicts
    races = sorted(race_numbers)
    blended_probs = {}
    market_probs = {}
    sorted_horses = {}
    
    for race_num in races:
        rkey = f"Race {race_num}"
        race_data = df_pick[df_pick[race_col] == race_num]
        
        blended_probs[rkey] = dict(zip(race_data[horse_col], race_data['blended_prob']))
        market_probs[rkey] = dict(zip(race_data[horse_col], race_data[ml_prob_col_used]))
        sorted_horses[rkey] = sorted(blended_probs[rkey].items(), 
                                      key=itemgetter(1), reverse=True)
    
    # Minimum selections per race
    min_selection = [1] * bet_type
    if min_prob is not None:
        for i, race in enumerate(sorted(blended_probs.keys())):
            cum = 0
            count = 0
            for _, prob in sorted_horses[race]:
                cum += prob
                count += 1
                if cum >= min_prob:
                    break
            min_selection[i] = max(1, count)
    
    max_horses = [len(sorted_horses[r]) for r in sorted(blended_probs.keys())]
    
    # Generate all allocations
    all_allocations = []
    for total in range(sum(min_selection), sum(max_horses) + 1):
        base = min_selection.copy()
        remaining = total - sum(base)
        
        def distribute(current, rem, idx=0):
            if idx == len(current) - 1:
                if current[idx] + rem <= max_horses[idx]:
                    alloc = current.copy()
                    alloc[idx] += rem
                    all_allocations.append(alloc)
                return
            for i in range(rem + 1):
                if current[idx] + i <= max_horses[idx]:
                    c = current.copy()
                    c[idx] += i
                    distribute(c, rem - i, idx + 1)
        
        distribute(base, remaining)
    
    # Evaluate each allocation
    results = []
    race_list = sorted(blended_probs.keys())
    net_pool = pool_size * (1 - takeout_rate)
    
    for allocation in all_allocations:
        cost = int(np.prod(allocation))
        
        if max_cost is not None and cost > max_cost:
            continue
        
        total_prob = 1.0
        raw_market_prob = 1.0
        selections = {}
        
        for i, race in enumerate(race_list):
            top = sorted_horses[race][:allocation[i]]
            race_prob = sum(p for _, p in top)
            market_p = sum(market_probs[race].get(h, 0) for h, _ in top)
            
            total_prob *= race_prob
            raw_market_prob *= market_p
            selections[race] = [h for h, _ in top]
        
        if min_total_prob is not None and total_prob < min_total_prob:
            continue
        
        if raw_market_prob > 0:
            expected_payout = (total_prob / raw_market_prob) * net_pool
            ev = expected_payout - cost
        else:
            ev = -cost
        
        results.append((cost, ev, total_prob, selections))
    
    # Sort by EV descending
    results.sort(key=lambda x: x[1], reverse=True)
    
    # Efficient frontier: best EV per cost level
    frontier = []
    best_ev = {}
    for r in results:
        cost = r[0]
        if cost not in best_ev or r[1] > best_ev[cost]:
            best_ev[cost] = r[1]
            frontier.append(r)
    
    frontier.sort(key=lambda x: x[0])
    
    if debug:
        print(f"\nEvaluated {len(all_allocations)} allocations, {len(results)} valid")
        print(f"Frontier: {len(frontier)} points")
    
    # Sample frontier points
    if frontier and budget_steps > 0 and budget_steps < len(frontier):
        indices = np.linspace(0, len(frontier) - 1, budget_steps, dtype=int)
        return [frontier[i] for i in indices]
    
    return frontier
 
 
def print_pick_tickets(frontier, bet_type=5, top_n=10):
    """Print top ticket options from the frontier."""
    sorted_f = sorted(frontier, key=lambda x: x[1], reverse=True)
    
    print(f"\n{'='*65}")
    print(f"TOP PICK {bet_type} TICKETS BY EXPECTED VALUE")
    print(f"{'='*65}")
    
    for i, (cost, ev, prob, selections) in enumerate(sorted_f[:top_n]):
        roi = ev / cost * 100 if cost > 0 else 0
        
        print(f"\nOption {i+1}: ${cost} cost | EV ${ev:+,.0f} | "
              f"Hit {prob:.2%} | ROI {roi:+.0f}%")
        print("-" * 50)
        
        for race_key in sorted(selections.keys()):
            race_num = int(race_key.split()[1])
            horses = selections[race_key]
            print(f"  R{race_num}: {', '.join(str(int(h)) for h in horses)}")
 
 
def print_race_probabilities(df, race_numbers, 
                              projected_prob_col='projected_win_probability',
                              ml_prob_col=None, blend_factor=0.5,
                              race_col='race', horse_col='horse_number'):
    """Print probability comparison table for pick ticket races."""
    
    has_ml = (ml_prob_col is not None and 
              ml_prob_col in df.columns and 
              df[ml_prob_col].notna().any())
    
    for race_num in race_numbers:
        race = df[df[race_col] == race_num].copy()
        
        if has_ml:
            race['blended'] = (blend_factor * race[projected_prob_col] + 
                               (1 - blend_factor) * race[ml_prob_col])
            total = race['blended'].sum()
            if total > 0:
                race['blended'] /= total
            race = race.sort_values('blended', ascending=False)
        else:
            race = race.sort_values(projected_prob_col, ascending=False)
        
        print(f"\n--- Race {race_num} ({len(race)} horses) ---")
        
        if has_ml:
            print(f"{'#':>3} {'Model':>7} {'ML':>7} {'Blend':>7}")
            for _, row in race.iterrows():
                print(f"{int(row[horse_col]):>3} "
                      f"{row[projected_prob_col]:>6.1%} "
                      f"{row[ml_prob_col]:>6.1%} "
                      f"{row['blended']:>6.1%}")
        else:
            print(f"{'#':>3} {'Model':>7}")
            for _, row in race.iterrows():
                print(f"{int(row[horse_col]):>3} {row[projected_prob_col]:>6.1%}")
 
 

In [155]:
if __name__ == '__main__':
    
    DATE = '4_3_26'
    TRACK = 'keeneland'
    STD_DEV = 4.0
    DF = 1.5
    
    # --- Build ensemble ---
    print("Building ensemble probabilities...")
    df = load_ai_picks(DATE, '../data/ai_picks')
    df = compute_composite_beyer(df)
    df = beyer_to_win_probabilities(
        df, std_dev=STD_DEV, degrees_of_freedom=DF, num_simulations=200000
    )
    
    # --- Pick 5: Races 1-5 ---
    print("\n" + "="*65)
    print("EARLY PICK 5: RACES 1-5")
    print("="*65)
    
    early_races = [1, 2, 3, 4, 5]
    print_race_probabilities(df, early_races)
    
    frontier_early = optimize_pick_tickets(
        df, early_races,
        pool_size=80000,
        takeout_rate=0.22,
        max_cost=2500,
        min_total_prob=0.1,
        budget_steps=20
    )
    print_pick_tickets(frontier_early, bet_type=5, top_n=8)
    
    # # --- Pick 5: Races 6-10 ---
    # print("\n" + "="*65)
    # print("LATE PICK 5: RACES 6-10")
    # print("="*65)
    
    # late_races = [6, 7, 8, 9, 10]
    # print_race_probabilities(df, late_races)
    
    # frontier_late = optimize_pick_tickets(
    #     df, late_races,
    #     pool_size=150000,  # Late pick 5 usually bigger pool
    #     takeout_rate=0.22,
    #     max_cost=2500,
    #     min_total_prob=0.01,
    #     budget_steps=20
    # )
    # print_pick_tickets(frontier_late, bet_type=5, top_n=8)
    
    # # --- Pick 3: Races 8-10 (Late Pick 3 at Keeneland) ---
    # print("\n" + "="*65)
    # print("LATE PICK 3: RACES 8-10")
    # print("="*65)
    
    # p3_races = [8, 9, 10]
    
    # frontier_p3 = optimize_pick_tickets(
    #     df, p3_races,
    #     pool_size=50000,
    #     takeout_rate=0.22,
    #     max_cost=500,
    #     min_total_prob=0.02,
    #     budget_steps=15
    # )
    # print_pick_tickets(frontier_p3, bet_type=3, top_n=8)

Building ensemble probabilities...

EARLY PICK 5: RACES 1-5

--- Race 1 (9 horses) ---
  #   Model
  6  52.1%
  8  26.5%
  2   6.8%
  5   5.1%
  1   2.5%
  3   2.1%
  7   1.9%
  4   1.7%
  9   1.4%

--- Race 2 (12 horses) ---
  #   Model
  5  47.5%
  4   9.5%
  9   8.0%
  7   5.9%
 12   5.6%
 11   4.4%
  8   4.4%
  6   3.6%
  1   3.4%
  3   3.4%
 10   2.5%
  2   1.8%

--- Race 3 (9 horses) ---
  #   Model
  4  45.7%
  8  11.5%
  3   9.5%
  1   8.8%
  2   6.9%
  7   5.6%
  5   4.7%
  9   4.7%
  6   2.6%

--- Race 4 (16 horses) ---
  #   Model
  3  18.7%
  9  16.7%
  5  12.3%
 14   9.4%
  8   9.3%
  7   5.1%
  4   5.1%
 15   4.9%
 11   4.3%
 10   3.7%
 12   3.0%
  2   2.8%
  6   1.6%
 16   1.1%
 13   1.0%
  1   0.9%

--- Race 5 (7 horses) ---
  #   Model
  2  32.0%
  4  29.0%
  3  15.3%
  7   9.6%
  6   7.0%
  1   5.5%
  5   1.6%
No ML odds available. Using model probabilities for ticket construction.
EV calculations will assume fair market (no overlay detection).

TOP PICK 5 TICKETS BY 

In [157]:
import os
import requests
import argparse

import numpy as np
import pandas as pd

from tqdm import tqdm
from bs4 import BeautifulSoup
url = "https://entries.horseracingnation.com/entries-results/keeneland/2026-04-03"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

entry_tables = soup.find_all('table', {'class': 'table-entries'})
rows = entry_tables[0].find_all('tr')

# Dump the raw HTML of the first entry row
first_row = rows[0]
cols = first_row.find_all('td')
print(f"Number of columns: {len(cols)}")
for i, col in enumerate(cols):
    print(f"\n=== Col {i} ===")
    print(col.prettify()[:400])

Number of columns: 6

=== Col 0 ===
<td data-label="Program Number: 1">
 <img alt="1" src="/static/images/lanes/1_pp-ef158b3fcc5af2672916458cfc208bcb.png?vsn=d" title="1">
 </img>
</td>


=== Col 1 ===
<td data-label="Post Position">
 1
</td>


=== Col 2 ===
<td data-label="Horse / Sire">
 <h4>
  <a class="horse-link" href="https://www.horseracingnation.com/horse/Fortune_King">
   Fortune King
  </a>
  <span class="small">
  </span>
 </h4>
 <p>
  Catalina Cruiser
 </p>
</td>


=== Col 3 ===
<td data-label="Trainer / Jockey">
 <p>
  Sergio Donjuan
 </p>
 <p>
  Summer Pauly
 </p>
</td>


=== Col 4 ===
<td class="table-entries-scratch-col" data-label="Scratched?">
</td>


=== Col 5 ===
<td data-label="Morning Line Odds">
 <p>
  15/1
 </p>
 <p class="table-entries-scratch-sm">
  <abbr title="">
  </abbr>
 </p>
</td>

